In [1]:
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

import numpy as np 
import tensorflow as tf 
from tensorflow.keras import * 

from tensorflow.keras.layers import * 
from tensorflow.keras.models import load_model, save_model 
from tensorflow.keras import Model
import tensorflow.keras.utils as utils
from keras.datasets import mnist 

from includes.fully_connected_opt_weight_generation import *

ModuleNotFoundError: No module named 'includes'

### Original Floating-point Model 

In [72]:
def Net(): 
    # inputs = Input(shape=x_train.shape[1:])
    inputs = Input(shape=(28, 28, 1))
    x = Conv2D(filters=4, kernel_size=(3,3), strides=(1,1), padding='same')(inputs)
    x = ReLU()(x)
    x = MaxPool2D((2,2),strides=(2,2), padding="same")(x)

    x = Flatten()(x)
    x = Dense(20)(x)

    x = ReLU()(x)
    x = Dense(10)(x)
    predictions = Softmax()(x)
    return Model(inputs=inputs, outputs=predictions)

In [73]:
epochs = 5 
num_classes = 10 
(x_train, y_train_num), (x_test, y_test_num) = mnist.load_data() 
x_train = x_train.reshape(x_train.shape[0], x_train.shape[1], x_train.shape[2], 1)
x_test = x_test.reshape(x_test.shape[0], x_test.shape[1], x_test.shape[2], 1)
y_train = utils.to_categorical(y_train_num, num_classes)
y_test = utils.to_categorical(y_test_num, num_classes)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

model = Net()
model.load_weights('neural_network_model.h5')

(60000, 28, 28, 1) (10000, 28, 28, 1) (60000, 10) (10000, 10)


In [74]:
model.summary()

Model: "model_37"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         [(None, 28, 28, 1)]       0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 28, 28, 4)         40        
_________________________________________________________________
re_lu_4 (ReLU)               (None, 28, 28, 4)         0         
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 14, 14, 4)         0         
_________________________________________________________________
flatten_2 (Flatten)          (None, 784)               0         
_________________________________________________________________
dense_4 (Dense)              (None, 20)                15700     
_________________________________________________________________
re_lu_5 (ReLU)               (None, 20)                0  

In [75]:
length = y_test.shape[0]
positives = 0
res = 0 
for i in range(length): 
    pred = model(x_test[i:i+1]).numpy().squeeze()
    y_test = (y_test > 0.5)
    pred_cast = (pred > 0.5)
    if i == 5 or i == 10 or i == 15: 
        print("y_test   :", y_test[i])
        print("pred     :", pred)
        print("pred_cast:", pred_cast)
        print("res    :", y_test[i]==pred)
        print('\n')

    positives = np.sum(y_test[i] == pred)
    if positives == 10: 
        res += 1
        
print("Accuracy:", res/length)

y_test   : [False  True False False False False False False False False]
pred     : [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
pred_cast: [False  True False False False False False False False False]
res    : [ True  True  True  True  True  True  True  True  True  True]


y_test   : [ True False False False False False False False False False]
pred     : [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
pred_cast: [ True False False False False False False False False False]
res    : [ True  True  True  True  True  True  True  True  True  True]


y_test   : [False False False False False  True False False False False]
pred     : [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
pred_cast: [False False False False False  True False False False False]
res    : [ True  True  True  True  True  True  True  True  True  True]


Accuracy: 0.867


In [100]:
"""
quantize numpy array
return:
x_q: quantized array
x_fp: reverse quantized array to float
"""
def quantize_array(x, bit_depth=8):
    min_x = x.min() 
    max_x = x.max()

    #find number of integer bits to represent this range
    int_bits = int(np.ceil(np.log2(max(abs(min_x),abs(max_x)))))
    
    if int_bits < 0: int_bits = 0

    frac_bits = bit_depth - 1 - int_bits #remaining bits are fractional bits (1-bit for sign)

    #floating point weights are scaled and rounded to [-2^(bit-1),2^(bit-1)-1], which are used in 
    #the fixed-point operations on the actual hardware (i.e., microcontroller)
    x_q = np.round(x*(2**frac_bits))

    # To quantify the impact of quantized weights, scale them back to
    # original range to run inference using quantized weights
    x_fp = x_q/(2**frac_bits)
    
    return x_q, x_fp, int_bits, frac_bits


"""
compute frac_bits based on max and min list
used for activation(feature map) quantization
"""
def min_max_quantize(max_list, min_list, bit_depth=8):
    frac_list = []
    for i in range(len(max_list)):
        int_bits = int(np.ceil(np.log2(max(abs(min_list[i]),abs(max_list[i])))))
        if int_bits < 0: int_bits = 0
        frac_bits = bit_depth - 1 - int_bits #remaining bits are fractional bits (1-bit for sign)
        frac_list.append(frac_bits)
    return frac_list

### Quantized Model

In [103]:
def QNet(bit=32, shift_list=[]):
    shift = False 
    if bit != 32: 
        shift = True 
    
    # input shape = (28, 28, 1)
    input = Input(shape=x_train.shape[1:], name='input')
    x = Conv2D(filters=4, kernel_size=(3,3), strides=(1,1), padding='same', name='conv1')(input)
    if shift:
        x = x / 2**shift_list[0]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)  
    x = ReLU(name='relu1')(x)

    x = MaxPool2D((2,2), strides=(2,2), padding='same', name='max_pool')(x)
    x = Flatten(name='flatten')(x)

    x = Dense(20, name='fc1')(x)
    if shift:
        x = x / 2**shift_list[1]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)
    x = ReLU(name='relu2')(x)
    
    x = Dense(10, name='fc2')(x)

    if shift: 
        x = x / 2**shift_list[2]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)
    else:
        x_out = x
    x_out = Softmax(name='softmax')(x)
    return Model(input, x_out)


# To check if the layer name went in correctly
model = QNet()
model.load_weights('neural_network_model.h5')
layer_names = [] 
complete_layers_name = [] 

for layer in model.layers: 
    if len(layer.get_weights()) != 0:
        layer_names.append(layer.name)
    complete_layers_name.append(layer.name)

complete_layers_name.remove('input')

layer_num = len(layer_names)
complete_layer_num = len(complete_layers_name)
print("Layer (With Weights) Names: ", layer_names)
print("TOTAL Layer Names: ", complete_layers_name)

Layer (With Weights) Names:  ['conv1', 'fc1', 'fc2']
TOTAL Layer Names:  ['conv1', 'relu1', 'max_pool', 'flatten', 'fc1', 'relu2', 'fc2', 'softmax']


In [104]:
"""
 intermediate feature map retrieval 
 for max and min value acquisition and computing quantization format
"""
intermediate_output = [model.get_layer('input').output]
print("Intermediate Output::", intermediate_output)

for i in range(layer_num):
    intermediate_output.append(model.get_layer(layer_names[i]).output)
print("Intermediate Output::", intermediate_output)
# intermediate output of input and other weighted layers are stored in a list

intermediate_layer_model = Model(inputs=model.input, outputs=intermediate_output)
print(intermediate_layer_model)
print(intermediate_output)
print('\n')
#-------------------------------

# Get output from all layers
complete_intermediate_output = []
for i in range(complete_layer_num):
    complete_intermediate_output.append(model.get_layer(complete_layers_name[i]).output)
print("Complete Intermediate Output::", complete_intermediate_output)

complete_intermediate_layer_model = Model(inputs=model.input, outputs=complete_intermediate_output)
print(complete_intermediate_layer_model)

Intermediate Output:: [<KerasTensor: shape=(None, 28, 28, 1) dtype=float32 (created by layer 'input')>]
Intermediate Output:: [<KerasTensor: shape=(None, 28, 28, 1) dtype=float32 (created by layer 'input')>, <KerasTensor: shape=(None, 28, 28, 4) dtype=float32 (created by layer 'conv1')>, <KerasTensor: shape=(None, 20) dtype=float32 (created by layer 'fc1')>, <KerasTensor: shape=(None, 10) dtype=float32 (created by layer 'fc2')>]
[<KerasTensor: shape=(None, 28, 28, 1) dtype=float32 (created by layer 'input')>, <KerasTensor: shape=(None, 28, 28, 4) dtype=float32 (created by layer 'conv1')>, <KerasTensor: shape=(None, 20) dtype=float32 (created by layer 'fc1')>, <KerasTensor: shape=(None, 10) dtype=float32 (created by layer 'fc2')>]


Complete Intermediate Output:: [<KerasTensor: shape=(None, 28, 28, 4) dtype=float32 (created by layer 'conv1')>, <KerasTensor: shape=(None, 28, 28, 4) dtype=float32 (created by layer 'relu1')>, <KerasTensor: shape=(None, 14, 14, 4) dtype=float32 (created by 

In [105]:
print(x_test[1].shape)
print(x_test[10:11].shape)

(28, 28, 1)
(1, 28, 28, 1)


In [107]:
"""
Extract the maximum and minimum value for 
the output of each layer specified in the previous cell
"""
frac_bits_output_list = [] 
max_list = []
min_list = [] 
for i in range(length):
    pred = intermediate_layer_model(x_test[i:i+1])
    for i in range(len(pred)):
        pred_np = pred[i].numpy()
        max_np = pred_np.max()
        min_np = pred_np.min()
        if len(max_list) != len(pred):
            max_list.append(max_np)
            min_list.append(min_np)
        else:
            if max_np > max_list[i]:
                max_list[i] = max_np
            if min_np < min_list[i]:
                min_list[i] = min_np

print("Max list::", max_list)
print("Min list::", min_list)
print()
bit_depth = 16 
frac_bits_activation_list = min_max_quantize(max_list, min_list, bit_depth=bit_depth)

print("input: ", frac_bits_activation_list)
for i in range(layer_num):
    print(i+1, " ", layer_names[i], " ", frac_bits_activation_list[i+1])

Max list:: [255.0, 725.2552, 6643.3813, 5808.3994]
Min list:: [0.0, -697.64514, -7803.087, -5468.6914]

input:  [7, 5, 2, 2]
1   conv1   5
2   fc1   2
3   fc2   2


### Quantization (16 bits)


In [108]:
bit_depth = 16 
frac_bits_weight_list = [] 
frac_bits_bias_list = [] 
q_weight_list = [] 
q_bias_list = [] 

for i in range(layer_num):
    weight, bias = model.get_layer(layer_names[i]).get_weights()
    q_weight, f_weight, int_bits_weight, frac_bits_weight = quantize_array(weight, bit_depth=bit_depth)
    q_bias, f_bias, int_bits_bias, frac_bits_bias = quantize_array(bias, bit_depth=bit_depth)
    
    q_weight_list.append(q_weight)
    q_bias_list.append(q_bias)
    frac_bits_weight_list.append(frac_bits_weight)
    frac_bits_bias_list.append(frac_bits_bias)
    print('***********')
    print(layer_names[i] + '-- weight -- Q' + str(int_bits_weight) + '.' + str(frac_bits_weight))
    print(layer_names[i] + '-- bias -- Q' + str(int_bits_bias) + '.' + str(frac_bits_bias))    


***********
conv1-- weight -- Q1.14
conv1-- bias -- Q0.15
***********
fc1-- weight -- Q0.15
fc1-- bias -- Q0.15
***********
fc2-- weight -- Q0.15
fc2-- bias -- Q0.15


In [115]:
# computing bias shift-left and output shift-right
bias_shift_list = np.array(frac_bits_weight_list) + np.array(frac_bits_activation_list[0:-1]) - np.array(frac_bits_bias_list)
output_shift_list = np.array(frac_bits_weight_list) + np.array(frac_bits_activation_list[0:-1]) - np.array(frac_bits_activation_list[1:])  
print(bias_shift_list)
print(output_shift_list)

[6 5 2]
[16 18 15]


In [116]:
q_model = QNet(bit=bit_depth, shift_list=output_shift_list)

for i in range(layer_num):
    q_model.get_layer(layer_names[i]).set_weights([q_weight_list[i], q_bias_list[i] * (2**bias_shift_list[i])])

# evaluation 
length = y_test.shape[0]
positives = 0
for i in range(length):
    pred = q_model(np.round(x_test[i:i+1]*(2**frac_bits_activation_list[0])))
    pred_cast = pred > 0.5

    if i == 0 or i == 5 or i == 10:
        print("y_test   :", y_test[i])
        print("pred     :", pred)
        print("pred_cast:", pred_cast)
        print("res    :", y_test[i]==pred_cast)
        print('\n')
    positives += np.sum(y_test[i] == pred)
    if positives == 10:
        res += 1
    
print("Accuracy after quantization: ", res / length)

y_test   : [False False False False False False False  True False False]
pred     : tf.Tensor(
[[3.5326284e-24 2.3802664e-26 8.5330483e-17 5.1090889e-12 7.7811329e-20
  9.6026799e-24 3.9754500e-31 1.0000000e+00 8.5330483e-17 2.5436657e-13]], shape=(1, 10), dtype=float32)
pred_cast: tf.Tensor([[False False False False False False False  True False False]], shape=(1, 10), dtype=bool)
res    : tf.Tensor([[ True  True  True  True  True  True  True  True  True  True]], shape=(1, 10), dtype=bool)


y_test   : [False  True False False False False False False False False]
pred     : tf.Tensor(
[[1.1253297e-07 2.2602853e-06 3.0589635e-07 1.6701373e-05 1.0261679e-10
  1.5229682e-08 9.3574400e-14 1.1253297e-07 9.9998045e-01 1.5229682e-08]], shape=(1, 10), dtype=float32)
pred_cast: tf.Tensor([[False False False False False False False False  True False]], shape=(1, 10), dtype=bool)
res    : tf.Tensor([[ True False  True  True  True  True  True  True False  True]], shape=(1, 10), dtype=bool)


y_te

### Saving quantized weight as .h (Clang) file

In [117]:
"""
Reordering and then saving onto a header file
"""

reordered_weight_list = []
for i in range(len(q_weight_list)):
    # convolution layer
    if len(q_weight_list[i].shape) == 4:
        reordered_weight_list.append(np.moveaxis(q_weight_list[i], 2, 0))
    # dense layer
    else: 
        reordered_weight_list.append(np.moveaxis(q_weight_list[i], 1, 0))

# flatten the weight tensor 
reordered_weight_flatten_list = []
for i in range(len(q_weight_list)): 
    # convolution layer 
    if len(reordered_weight_list[i].shape) == 4:
        reordered_weight_flatten_list.append(reordered_weight_list[i].flatten('F'))
    # dense layer
    else: 
        reordered_weight_flatten_list.append(reordered_weight_list[i].flatten())

# save 
f = open('./quantized_weight.h', 'w')
for i in range(len(q_weight_list)): 
    print('********' + layer_names[i] + '********')
    print('Bias shift: ', bias_shift_list[i])
    print('Output shift: ', output_shift_list[i])
    print('Bias: ', q_bias_list[i].astype(np.int16))
    print('Weight: ', reordered_weight_flatten_list[i].astype(np.int16))
    
    f.write('#define ' + layer_names[i].upper() + '_WT {' + str(reordered_weight_flatten_list[i].astype(np.int16).tolist())[1:-1] + '}\n')
    f.write('#define ' + layer_names[i].upper() + '_BIAS {' + str(q_bias_list[i].astype(np.int16).tolist())[1:-1] + "}\n")
    f.write('#define ' + layer_names[i].upper() + '_BIAS_LSHIFT ' + str(bias_shift_list[i]) + "\n")
    f.write('#define ' + layer_names[i].upper() + '_OUT_RSHIFT ' + str(output_shift_list[i]) + "\n\n")
    
f.close()

********conv1********
Bias shift:  6
Output shift:  16
Bias:  [2201 1520  207 2458]
Weight:  [  8406   9357  -2454   4706   9241   3335   -397  10323   1326 -14350
  -2323   6958   2221   5810   7165   2512  11337   1327   6274    272
   2981   9652   6987   1289   7059  -3666   5606   1972 -11224 -17284
     11  -3605 -12717  15139  14448   9648]
********fc1********
Bias shift:  5
Output shift:  18
Bias:  [ 4317 -3337  -210  -304  -182 -3250 -1273  5871  5256 -2090  2171   209
  -403 -4364  4434  4922 -2861 -1042 -2744  2424]
Weight:  [4022 4356 1976 ... 5535 3545 5421]
********fc2********
Bias shift:  2
Output shift:  15
Bias:  [ -590  6638 -1385  3822 -4798  3286  1777  2647 -9010   270]
Weight:  [  7678  -9049  18223   1920  -3680 -11178  -8527 -13172 -14892   2144
 -14143   5668  11519  -5586   3723 -21161  -2009  17637  -3384   9497
  12074 -10996 -17808  12118  10990  -3126 -16310   6093 -20622   3895
   6323  16650  -9424   4097  16304  12577 -18344  -5193   2306   7912
  11850

In [ ]:
q_reordered_weight_list = []

for i in range(layer_num): 
    print(q_weight_list[i].shape)
    input_dim, output_dim = q_weight_list[i].shape
    q_expanded_weight_list = np.reshape(q_weight_list[i], (output_dim, input_dim, 1, 1))
    reordered_weights = convert_to_x4_q15_weights(q_expanded_weight_list)
    final_weights = np.reshape(reordered_weights, (output_dim, input_dim))
    q_reordered_weight_list.append(final_weights)

# flatten the weight tensor 
weight_flatten_list = []
for i in range(len(q_reordered_weight_list)): 
    # convolution layer 
    if len(q_reordered_weight_list[i].shape) == 4:
        weight_flatten_list.append(q_reordered_weight_list[i].flatten('F'))
    # dense layer
    else: 
        weight_flatten_list.append(q_reordered_weight_list[i].flatten())


f = open('./opt_quantized_weight.h', 'w')
for i in range(len(q_weight_list)): 
    print('********' + layer_names[i] + '********')
    print('Bias shift: ', bias_shift_list[i])
    print('Output shift: ', output_shift_list[i])
    print('Bias: ', q_bias_list[i].astype(np.int16))
    print('Weight: ', weight_flatten_list[i].astype(np.int16))
    
    f.write('#define ' + layer_names[i].upper() + '_WT {' + str(reordered_weight_flatten_list[i].astype(np.int16).tolist())[1:-1] + '}\n')
    f.write('#define ' + layer_names[i].upper() + '_BIAS {' + str(q_bias_list[i].astype(np.int16).tolist())[1:-1] + "}\n")
    f.write('#define ' + layer_names[i].upper() + '_BIAS_LSHIFT ' + str(bias_shift_list[i]) + "\n")
    f.write('#define ' + layer_names[i].upper() + '_OUT_RSHIFT ' + str(output_shift_list[i]) + "\n\n")

In [118]:
intermediate_output = []
intermediate_output.append(q_model.get_layer('input').output)

for i in range(layer_num):
    intermediate_output.append(q_model.get_layer(layer_names[i]).output)

q_intermediate_layer_model = Model(inputs=q_model.input, outputs=intermediate_output)
sample_output = q_intermediate_layer_model.predict(np.round(x_test[0:1]*(2**frac_bits_activation_list[0])))

# save with weight reordering
f = open("./sample_input_output.h", "w")
temp = np.squeeze(sample_output[0], axis=0)
temp = temp.transpose(2, 0, 1)
temp = temp.flatten("F")
f.write("#define " + "INPUT_DATA {" + str(temp.astype(np.int16).tolist())[1:-1] + "}\n")

for i in range(layer_num):
    temp = np.squeeze(sample_output[i+1], axis=0)
    print(temp.shape)
    if len(temp.shape) == 3:
        temp = temp.transpose(2, 0, 1)
        temp = temp.flatten("F")
    
    temp = temp.astype(np.int32) >> output_shift_list[i]
    f.write("#define " + layer_names[i].upper() + " {" + str(temp.tolist())[1:-1] + "}\n")

f.close()
print("finished reordering and writing input and output to file")


(28, 28, 4)
(20,)
(10,)
finished reordering and writing input and output to file


In [122]:
# complete output map 
# complete_output_shift_list = [16, 16, 16, 16, 18, 18, 15, 15]
# complete_intermediate_output = []
# complete_intermediate_output.append(q_model.get_layer(layer_names[i]).output)

# for i in range(complete_layer_num):
#     complete_intermediate_output.append(q_model.get_layer(complete_layers_name[i]).output)

# q_intermediate_layer_model = Model(inputs=q_model.input, outputs=complete_intermediate_output)
# sample_output = q_intermediate_layer_model.predict(np.round(x_test[0:1] * (2**frac_bits_activation_list[0])))

# f = open("./complete_sample_input_output.h", "w")
# temp = np.squeeze(sample_output[0], axis=0)
# temp = temp.transpose(2, 0, 1)
# temp = temp.flatten("F")
# f.write("#define " + "INPUT_DATA {" + str(temp.astype(np.int16).tolist())[1:-1] + "}\n")

# for i in range(complete_layer_num):
#     temp = np.squeeze(sample_output[i+1], axis=0)
#     print(temp.shape)
#     if len(temp.shape) == 3:
#         temp = temp.transpose(2, 0, 1)
#         temp = temp.flatten("F")
#     temp = temp.astype(np.int32) >> complete_output_shift_list[i]
#     f.write("#define " + complete_layers_name[i].upper() + " {" + str(temp.tolist())[1:-1] + "}\n")

# f.close()
# print("finished reordering and writing input and output to file")

In [121]:
for i in range(0, len(intermediate_output)):
    print(intermediate_output[i].shape)

for i in range(0, len(sample_output)):
    print(sample_output[i].shape)

(None, 28, 28, 1)
(None, 28, 28, 4)
(None, 20)
(None, 10)
(1, 10)
(1, 28, 28, 4)
(1, 28, 28, 4)
(1, 14, 14, 4)
(1, 784)
(1, 20)
(1, 20)
(1, 10)
(1, 10)
